In [1]:
q1="I just discovered the course, can I still join?"
q2="I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
v1=model.encode(q1)
v1.shape

(384,)

In [4]:
v2=model.encode(q2)
v2.shape

(384,)

In [5]:
doc="You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
docv = model.encode(doc)

In [6]:
v1.dot(docv)

np.float32(0.39572883)

In [7]:
v2.dot(docv)

np.float32(0.463655)

In [8]:
from ingest import load_faq_data

documents=load_faq_data()

In [9]:
len(documents)

1350

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
texts[1000]

"How do you find the correlation matrix? First, you have to consider whether the data is numerical or categorical. If it’s numerical, you can correlate it directly. If it’s categorical, you can find the correlations indirectly by vectorizing the data using One-Hot encoding or a similar method.\n\nTo determine if data is numerical, check the `dtypes` of the DataFrame. Data types such as integer and float are numerical, while types such as objects are categorical. You can correlate the numerical data by specifying which columns are numerical and using that as input to a correlation matrix.\n\nExample:\n\n```python\nnumerical = ['tenure', 'monthlycharges', 'totalcharges']\n\ncorrelation_matrix = df[numerical].corr()\nprint(correlation_matrix)\n```"

In [12]:
from tqdm.auto import tqdm

batch_size=50

vectors=[]

for i in tqdm(range(0, len(texts), batch_size)):
    batch=texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
scores=[]

for i in range(len(vectors)):
    score=v1.dot(vectors[i])
    scores.append(score)

In [14]:
import numpy as np
X = np.array(vectors)
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091588, -0.06990605, -0.0093188 , ..., -0.00044342,
        -0.0128573 ,  0.01426918],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1350, 384), dtype=float32)

In [15]:
scores=X.dot(v1)

In [16]:
idx=np.argmax(scores)
idx,scores[idx]

(np.int64(538), np.float32(0.8317791))

In [17]:
documents[100]

{'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Postgres, pgAdmin & Python ingestion',
 'question': 'Postgres: bind: address already in use',
 'answer': 'When attempting to start the Docker Postgres container, you may encounter the error message:\n\n```\nError - postgres port is already in use.\n```\n\n**Option 1: Identify and Stop the Service**\n\n1. Determine which service is using the port by running:\n   \n   ```bash\n   sudo lsof -i :5432\n   ```\n   \n2. Stop the service that is using the port:\n   \n   ```bash\n   sudo service postgresql stop\n   ```\n\n**Option 2: Map to a Different Port**\n\nFor a more long-term solution, consider mapping to a different port:\n\n- Map local port 5433 to container port 5432 in your Docker configuration (`Dockerfile` or `docker-compose.yml`).\n- If using a VM, ensure that port 5433 is forwarded in the host machine configuration.\n\nThis approach prevents conflicts and allows the Docker Postgres container to run without interruptio

In [18]:
top5=np.argsort(scores)[-5:]
top5=top5[::-1]

scores[top5]

array([0.8317791 , 0.6845695 , 0.61755615, 0.60880876, 0.58479667],
      dtype=float32)

In [19]:
for i in top5:
    print(scores[i])
    print(documents[i])
    print()

0.8317791
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.6845695
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'doc_id': '41aabbd7c5'}

0.61755615
{'course': 'mlops-zoomcamp', 'section': 'Ge

In [20]:
from minsearch import VectorSearch

vecindex=VectorSearch(keyword_fields=["course"])
vecindex.fit(X, documents)

In [21]:
question = "I just dicovered the course. Can I still join it?"
que_vector=model.encode(question)

results=vecindex.search(que_vector, num_results=3)

In [22]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [23]:
results2=vecindex.search(
    que_vector,
    filter_dict={"course":"llm-zoomcamp"},
    num_results=5
)

In [24]:
results2[3]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'Can I run the course locally instead of Codespaces?',
 'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.',
 'doc_id': 'aa310de435'}

In [25]:
vecindex.search(v2,num_results=1)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'}]

In [26]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [27]:
from ingest import load_faq_data, build_index

documents=load_faq_data()
index=build_index(documents)

In [28]:
from rag_helper import RAGbase

assistant=RAGbase(
    index=index,
    llm_client=openai_client
)

In [29]:
q3="Can I use other LLMs?"

In [30]:
assistant.rag(q3)

'Yes. You can use other models/providers, including OpenAI, Gemini, Groq, OpenRouter, Azure OpenAI, local models, or another provider.\n\nIf you use a non-OpenAI provider, you may need to adapt the code because tool schemas, response formats, and tokenizers can differ.'

Expading RAGbase to Vector search

In [31]:
class RAGVector(RAGbase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [32]:
vec_assistant=RAGVector(
    embedder=model,
    index=vecindex,
    llm_client=openai_client
)

In [33]:
vec_assistant.rag(q2)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [34]:
vec_assistant.rag(q3)

'Yes. The recommended model is not mandatory—you can use OpenAI, Gemini, Groq, OpenRouter, Azure OpenAI, local models, or another provider. You may need to adapt the code for your provider because response formats, tool schemas, and tokenizers can differ.'

SQLite approach

In [35]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

In [36]:
vs_index=VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [42]:
vs_index.fit(vectors,documents)

In [43]:
#redefining RAGVector function

class RAGVector(RAGbase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [44]:
v_assistant=RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client
)

In [55]:
v_assistant.rag(q2)

'Yes — you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [46]:
vs_index.close()

In [ ]:
#reopening

vs_index=VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)